In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SeverityModeling") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

full_df = spark.read.parquet("../data/processed/trips_with_disruptions")

severity_df = full_df.filter(F.col("is_disrupted") == 1)

print("Disrupted trip rows:", severity_df.count())
severity_df.groupBy("severity").count().show()

26/07/14 16:43:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

Disrupted trip rows: 180949
+--------+------+
|severity| count|
+--------+------+
|  severe|135967|
|moderate| 38859|
|   minor|  6123|
+--------+------+



In [2]:
# each real disruption has a distinct combination of reason + overlap_duration_minutes,
# which lets us identify which of the 24 original disruptions each trip belongs to —
# needed to do a proper disruption-level train/test split rather than trip-level,
# since trip-level splitting would leak the same disruption's characteristics
# across train and test (only 24 real disruptions exist, despite 180,949 trip rows)
distinct_disruptions = severity_df.select("reason", "planned", "severity", "overlap_duration_minutes").distinct()
print("Distinct disruption signatures found:", distinct_disruptions.count())
distinct_disruptions.orderBy("overlap_duration_minutes").show(30, truncate=False)

Distinct disruption signatures found: 14
+--------------+-------+--------+------------------------+
|reason        |planned|severity|overlap_duration_minutes|
+--------------+-------+--------+------------------------+
|roadworks     |true   |minor   |10874.983333333334      |
|roadworks     |true   |minor   |32819.98333333333       |
|roadworks     |false  |minor   |58632.98333333333       |
|roadworks     |true   |minor   |58739.98333333333       |
|roadworks     |false  |moderate|67146.98333333334       |
|roadworks     |true   |moderate|70059.98333333334       |
|vandalism     |true   |moderate|88486.98333333334       |
|roadworks     |true   |moderate|91979.98333333334       |
|routeDiversion|true   |moderate|100105.98333333334      |
|roadworks     |true   |moderate|101939.98333333334      |
|roadworks     |true   |moderate|122099.98333333334      |
|roadClosed    |true   |severe  |131039.98333333334      |
|roadworks     |true   |severe  |131039.98333333334      |
|routeDiversion

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# pull the 14 distinct signatures to pandas for a small, manual stratified split
signatures_pd = distinct_disruptions.toPandas()
signatures_pd["signature_id"] = range(len(signatures_pd))

# stratified split ensures every severity class appears in both train and test,
# which matters given how few distinct disruption signatures exist (14)
train_sigs, test_sigs = train_test_split(
    signatures_pd, test_size=0.3, stratify=signatures_pd["severity"], random_state=42
)

print("Train signatures:", len(train_sigs))
print(train_sigs["severity"].value_counts())
print("\nTest signatures:", len(test_sigs))
print(test_sigs["severity"].value_counts())

Train signatures: 9
severity
moderate    4
minor       3
severe      2
Name: count, dtype: int64

Test signatures: 5
severity
moderate    3
severe      1
minor       1
Name: count, dtype: int64


In [4]:
# tag each disruption signature with train/test membership, then join back
# to the full trip-level table so every trip inherits its disruption's assignment
train_sigs_spark = spark.createDataFrame(train_sigs[["reason", "planned", "severity", "overlap_duration_minutes"]]) \
    .withColumn("split", F.lit("train"))
test_sigs_spark = spark.createDataFrame(test_sigs[["reason", "planned", "severity", "overlap_duration_minutes"]]) \
    .withColumn("split", F.lit("test"))

split_assignment = train_sigs_spark.union(test_sigs_spark)

severity_df_split = severity_df.join(
    split_assignment,
    on=["reason", "planned", "severity", "overlap_duration_minutes"],
    how="inner"
)

train_severity_df = severity_df_split.filter(F.col("split") == "train")
test_severity_df = severity_df_split.filter(F.col("split") == "test")

print("Train trip rows:", train_severity_df.count())
print("Test trip rows:", test_severity_df.count())

print("\nTrain severity distribution:")
train_severity_df.groupBy("severity").count().show()
print("Test severity distribution:")
test_severity_df.groupBy("severity").count().show()

Train trip rows: 153909


Test trip rows: 27040

Train severity distribution:


+--------+------+
|severity| count|
+--------+------+
|   minor|  3615|
|moderate| 16459|
|  severe|133835|
+--------+------+

Test severity distribution:
+--------+-----+
|severity|count|
+--------+-----+
|  severe| 2132|
|moderate|22400|
|   minor| 2508|
+--------+-----+



# Building features for severity classification:

In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# time features, same as before — legitimate here since we're predicting severity
# given a disruption is already happening, not predicting occurrence
train_severity_df = train_severity_df.withColumn(
    "departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int")
).withColumn(
    "hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "is_weekend", F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

test_severity_df = test_severity_df.withColumn(
    "departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int")
).withColumn(
    "hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24)
).withColumn(
    "is_weekend", F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

# categorical: reason and planned status — these directly describe the disruption itself
categorical_cols = ["reason", "planned"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

train_severity_df = train_severity_df.withColumn("planned", F.col("planned").cast("string"))
test_severity_df = test_severity_df.withColumn("planned", F.col("planned").cast("string"))

# label indexing for severity (multi-class target)
label_indexer = StringIndexer(inputCol="severity", outputCol="label", handleInvalid="keep")

pipeline = Pipeline(stages=indexers + encoders + [label_indexer])
pipeline_model = pipeline.fit(train_severity_df)

train_encoded = pipeline_model.transform(train_severity_df)
test_encoded = pipeline_model.transform(test_severity_df)

feature_cols = ["hour_sin", "hour_cos", "is_weekend", "reason_ohe", "planned_ohe"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

train_final = assembler.transform(train_encoded)
test_final = assembler.transform(test_encoded)

train_final.select("features", "label", "severity").show(5, truncate=False)

+--------------------------------------------------------------+-----+--------+
|features                                                      |label|severity|
+--------------------------------------------------------------+-----+--------+
|[-0.49999999637300596,-0.8660254058784846,0.0,1.0,0.0,0.0,1.0]|2.0  |minor   |
|[0.5000000025907099,-0.8660254022886916,0.0,1.0,0.0,0.0,1.0]  |2.0  |minor   |
|[-0.7071067780135887,-0.7071067843595064,0.0,1.0,0.0,0.0,1.0] |2.0  |minor   |
|[0.5000000025907099,-0.8660254022886916,0.0,1.0,0.0,0.0,1.0]  |2.0  |minor   |
|[-0.5000000056995618,0.8660254004937951,1.0,1.0,0.0,0.0,1.0]  |2.0  |minor   |
+--------------------------------------------------------------+-----+--------+
only showing top 5 rows


# Training all three models on the multi-class severity target

In [7]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
import time

severity_models = {}
severity_training_times = {}

lr = LogisticRegression(featuresCol="features", labelCol="label")
start = time.time()
severity_models["Logistic Regression"] = lr.fit(train_final)
severity_training_times["Logistic Regression"] = time.time() - start
print(f"Logistic Regression trained in {severity_training_times['Logistic Regression']:.2f}s")

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)
start = time.time()
severity_models["Decision Tree"] = dt.fit(train_final)
severity_training_times["Decision Tree"] = time.time() - start
print(f"Decision Tree trained in {severity_training_times['Decision Tree']:.2f}s")

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, numTrees=100)
start = time.time()
severity_models["Random Forest"] = rf.fit(train_final)
severity_training_times["Random Forest"] = time.time() - start
print(f"Random Forest trained in {severity_training_times['Random Forest']:.2f}s")

26/07/14 16:51:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Logistic Regression trained in 16.91s


Decision Tree trained in 5.72s


Random Forest trained in 21.22s


In [8]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

severity_results = []

for name, model in severity_models.items():
    predictions = model.transform(test_final)

    acc = acc_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    severity_results.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted Precision": precision,
        "Weighted Recall": recall,
        "Weighted F1": f1,
        "Training Time (s)": severity_training_times[name]
    })

import pandas as pd
severity_results_df = pd.DataFrame(severity_results)
print(severity_results_df.to_string(index=False))

              Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1  Training Time (s)
Logistic Regression  0.078846            0.006217         0.078846     0.011525          16.910221
      Decision Tree  0.078846            0.006217         0.078846     0.011525           5.724710
      Random Forest  0.078846            0.006217         0.078846     0.011525          21.218445


In [9]:
best_predictions = severity_models["Logistic Regression"].transform(test_final)
best_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show(20)

[Stage 306:=================================================>       (7 + 1) / 8]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 2132|
|  1.0|       0.0|22400|
|  2.0|       0.0| 2508|
+-----+----------+-----+



In [10]:
# training set is 87% severe, which caused all models to just predict severe always —
# downsample severe in training so the model can't win by ignoring the other classes
minor_count = train_final.filter(F.col("severity") == "minor").count()
moderate_count = train_final.filter(F.col("severity") == "moderate").count()
severe_count = train_final.filter(F.col("severity") == "severe").count()

target_count = max(minor_count, moderate_count)  # cap severe down to roughly match the next-largest class
severe_fraction = min(1.0, target_count / severe_count)

severe_rows = train_final.filter(F.col("severity") == "severe").sample(fraction=severe_fraction, seed=42)
other_rows = train_final.filter(F.col("severity") != "severe")

train_final_balanced = severe_rows.union(other_rows)

print("Rebalanced train distribution:")
train_final_balanced.groupBy("severity").count().show()

Rebalanced train distribution:


[Stage 322:===================================================>   (15 + 1) / 16]

+--------+-----+
|severity|count|
+--------+-----+
|  severe|16480|
|   minor| 3615|
|moderate|16459|
+--------+-----+



In [11]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
import time

severity_models_balanced = {}
severity_training_times_balanced = {}

lr = LogisticRegression(featuresCol="features", labelCol="label")
start = time.time()
severity_models_balanced["Logistic Regression"] = lr.fit(train_final_balanced)
severity_training_times_balanced["Logistic Regression"] = time.time() - start
print(f"Logistic Regression trained in {severity_training_times_balanced['Logistic Regression']:.2f}s")

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)
start = time.time()
severity_models_balanced["Decision Tree"] = dt.fit(train_final_balanced)
severity_training_times_balanced["Decision Tree"] = time.time() - start
print(f"Decision Tree trained in {severity_training_times_balanced['Decision Tree']:.2f}s")

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, numTrees=100)
start = time.time()
severity_models_balanced["Random Forest"] = rf.fit(train_final_balanced)
severity_training_times_balanced["Random Forest"] = time.time() - start
print(f"Random Forest trained in {severity_training_times_balanced['Random Forest']:.2f}s")

Logistic Regression trained in 13.14s


Decision Tree trained in 7.11s


Random Forest trained in 11.88s


In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

severity_results_balanced = []

for name, model in severity_models_balanced.items():
    predictions = model.transform(test_final)

    acc = acc_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    severity_results_balanced.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted Precision": precision,
        "Weighted Recall": recall,
        "Weighted F1": f1,
        "Training Time (s)": severity_training_times_balanced[name]
    })

import pandas as pd
severity_results_balanced_df = pd.DataFrame(severity_results_balanced)
print(severity_results_balanced_df.to_string(index=False))

              Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1  Training Time (s)
Logistic Regression  0.670673            0.741644         0.670673     0.681661          13.142668
      Decision Tree  0.680104            0.742517         0.680104     0.688252           7.109433
      Random Forest  0.692493            0.743338         0.692493     0.696724          11.881989


In [13]:
best_severity_predictions = severity_models_balanced["Random Forest"].transform(test_final)
best_severity_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show(20)

[Stage 547:=================================================>       (7 + 1) / 8]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 2132|
|  1.0|       0.0| 5807|
|  1.0|       1.0|16593|
|  2.0|       0.0|   72|
|  2.0|       1.0| 2436|
+-----+----------+-----+



In [14]:
test_final.groupBy("severity").count().show()

+--------+-----+
|severity|count|
+--------+-----+
|  severe| 2132|
|moderate|22400|
|   minor| 2508|
+--------+-----+

